In [ ]:
#Mount to Google Colab


Enable_3D_visualization = True   
from __future__ import division
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os
import subprocess
from operator import itemgetter
import csv
import pandas as pd
from collections import Counter
import itertools
from tabulate import tabulate
import os
import re
import datetime

import json
import glob
import os
import subprocess
import sys
import re


from tkinter import Tk

try:
  from google.colab import drive
  drive.mount('/content/drive')

  cossmos_work_dir = '/content/drive/Shareddrives/CoSSMos_database'

  if not os.path.exists(cossmos_work_dir):
      print('Working directory (%s) doesn\'t exist, contact Dr.Brent Znosko to get access permission' % work_dir)
      exit(-1)
  %cd /content/drive/Shareddrives/CoSSMos_database

  work_dir = '/content/drive/MyDrive/AML/AML_Project'

  %mkdir -p $work_dir


  # install biopython
  %shell pip install Bio

  '''
  def setup_3Dvisualization():
    !pip install ipywidgets
    !pip install rdkit-pypi
    !pip install py3Dmol # 3D Molecular Visualizer
    ! apt-get install pymol

    !wget -c https://repo.anaconda.com/miniconda/Miniconda3-py37_4.9.2-Linux-x86_64.sh
    !chmod +x Miniconda3-py37_4.9.2-Linux-x86_64.sh
    !time bash ./Miniconda3-py37_4.9.2-Linux-x86_64.sh -b -f -p /usr/local

    !time conda install -q -y -c conda-forge rdkit
    !time conda install -q -y -c openbabel openbabel

  setup_3Dvisualization()
  '''
  print("\nCurrent working directory: ", os.getcwd())

except subprocess.CalledProcessError:
  print(captured)
  raise

In [ ]:
#set up working directory

Working_directory = 'Hairpin_4'  #@param {type:"string"}

work_dir = '/content/drive/MyDrive/AML/AML_Project/'+Working_directory

!mkdir -p $work_dir

if not os.path.exists(work_dir):
    print('Working directory (%s) doesn\'t exist, contact Dr.Brent Znosko to get access permission' % work_dir)
    exit(-1)

#cur_work_dir = cossmos_work_dir+'/'+Working_directory

cur_work_dir = work_dir

print("Current working directory on google drive: ")
os.chdir(work_dir)

!pwd

In [ ]:
#Verify your file path of your CoSSMos file


CoSSMos_file_name = 'Bulge_1.txt'  #@param {type:"string"}


csvfile = work_dir+'/CoSSMos_web_outputs/'+CoSSMos_file_name

import os

if not os.path.exists(csvfile):
    print("Warning!! Failed to find ",csvfile, '! Please double-check the file')
    print("Double-check your file name, or the file you uploaded.")
    exit(-1)
else:
    print("Loaded successfully! Start processing ",csvfile, "\n\n")

os.chdir(work_dir)

In [ ]:
# Download and Clip the pdb files



#These next steps will use the CoSSMos output file to download individual .pdb files for each result in the CoSSMos search.

#The .pdb files contain data for the entire RNA strand. 


#!/usr/bin/env python
#This script will clip a loop and the first closing base pair
#from the a 2 closing base pair database


import os

if not os.path.exists(csvfile):
    print("Failed to find ",csvfile, '! Please double-check the file in Part 1.3')
    exit(-1)
else:
    print("Loaded successfully! Start processing ",csvfile, "\n\n")


from os import listdir
from subprocess import call
import copy
import shutil
import os.path
from os.path import isfile, join, basename, isdir


import numpy as np
import numexpr as ne
import csv

from itertools import combinations, groupby
from operator import itemgetter

from Bio.PDB import *
from optparse import OptionParser

#RangeSelect is the function that returns the coordinates of the loop
#and closing base pair for hairpins and bulges/internal loops with one chain
class RangeSelect(Select):
    def __init__(self, chain, resids2, positions_resids2_dict):
        self.chain = chain
        self.resids2 = resids2
        self.positions_resids2_dict = positions_resids2_dict
    def accept_residue(self, residue):
        if residue.parent.id == self.chain:
            rid = residue.get_id() # rid:  ('H_MG', 1644, ' ')
            rid = str(rid[1]) + str(rid[2])
            rid = rid.strip()
            found = 0
            if rid in self.resids2:
                #for key, value in positions_resids2_dict.iteritems():
                for key, value in self.positions_resids2_dict.items():
                    if rid == value:
                        #print("residue.id before: ",residue.id, residue.get_resname())
                        #residue.id = (rid[0], key, ' ') # this line cause HETATM
                        #print("to print: ",(residue.id[0], key, key))
                        residue.id = (residue.id[0], key, key)
                        #print("residue.id after: ",residue.id)
                found = 1
            return found

#BRangeSelect is the function that returns the coordinates for the loop and
#closing base pairs for bulges and internal loops with two chains
class BRangeSelect(Select):
    def __init__(self, chain, bchain, resids2, bresids2, positions_resids2_dict):
        self.chain = chain
        self.resids2 = resids2
        self.bchain = bchain
        self.bresids2 = bresids2
        self.positions_resids2_dict = positions_resids2_dict
    def accept_residue(self, residue):
        if residue.parent.id == self.chain:
            rid = residue.get_id()
            rid = str(rid[1]) + rid[2]
            rid = rid.strip()
            found = 0
            if rid in self.resids2:
                #for key, value in positions_resids2_dict.iteritems():
                for key, value in self.positions_resids2_dict.items():
                    if rid == value:
                        #residue.id = (rid[0], key, ' ') # this line cause HETATM
                        residue.id = (residue.id[0], key, key)
                found = 1
            return found
        if residue.parent.id == self.bchain:
            rid = residue.get_id()
            rid = str(rid[1]) + rid[2]
            rid = rid.strip()
            found = 0
            if rid in self.bresids2:
                for key, value in self.positions_resids2_dict.items():
                    if rid == value:
                        #residue.id = (rid[0], key, ' ')# this line cause HETATM
                        residue.id = (residue.id[0], key, key)
                found = 1
            return found

#This is the function that reads in the "#" delimited csv file
#If you have a different delimiter, you will need to replace the "#s" with
#the delimiter in your file
def read_csv(file):
    try:
      csvfile = open(file,'r')
    except:
      raise Exception("Failed to find ",file, '! Please double-check the file in Part 1.3')

    #replace csv library processing
    #if list of files was opened with excel it is
    #possible that some lines starte/stop with "
    #and then contents are not split correctly.
    lines = [line.strip() for line in csvfile]
    csvfile.close()

    header = lines[0].strip('"').split('\t')
    data = []
    for l in lines[1:]:
        data.append(l.strip('"').split('\t'))

    return header, data

#This is the funciton that checks to see if a folder is present
#If the folder is not present, it will create it
def check_folder(folder):
    if os.path.isdir(folder) == False:
        print('Folder \'%s\' does not exist...creating' % folder)
        try:
            os.makedirs(folder, exist_ok=True)
        except OSError as exception:
            if exception.errno != errno.EEXIST:
                raise
        if not os.path.isdir(folder):
          print("Failed to create folder ", folder)

def in_order(resids, beg, end):
    #determines if resid velus are in increments of one or not
    #print("here")
    if (end - beg + 1) == (len(resids.split())):
        return 1
    else:
        return 0

def renumber_resids(clipname,renumbered_dir):
    #copies original file from clipname to renumbered_dir
    #and then renumbered resids in increments of one
    base_fname=os.path.basename(clipname)
    shutil.copy(clipname,renumbered_dir+"\\"+base_fname)

    #read file
    f = open(clipname, 'r')
    lines = f.readlines()
    before = []
    num = []
    after = []
    for l in lines:
        before.append(l[0:23])
        num.append(l[23:26])
        after.append(l[26:])
    old_strings = sorted(set(num[:len(num)-2]))
    replace_string = []
    for i in range(len(old_strings)):
        replace_string.append(format(i+1,"3d"))
    for i in range(len(num)-2):
        ind = old_strings.index(num[i])
        num[i] = replace_string[ind]
    out = []
    for i in range(len(num)):
        out.append(before[i]+num[i]+after[i])
    f = open(clipname,"w")
    for line in out:
        f.write(line)
    f.close()

#Function that writes motif rejects to file
def write_rejects(fname,list_name):
    f = open(fname,"w")
    for line in list_name:
        f.write(line+"\n")
    f.close()

def atom_types_in_file(fname,unique):
    #returns list of atoms in order from file when unique=0
    #returns list of unique atoms in file when unuque=1, assume random order
    atoms = []
    with open(fname) as f:
        content = f.readlines()
    for line in content :
        record = line[0:6].strip()
        if record == 'ATOM' or record == "HETATM":
            atoms.append(line[13:16].strip())
            #print(line[13:16])
    if unique == 0:
        return atoms
    else:
        return list(set(atoms))

def purge_h_atoms(fname,h_removed_dir):
    #if H-atoms in file
    #(1) copy of file is made in same directory as *-H.pdb
    #(2) H's are removed from existing file and atom num bers are updated.
    base_fname = os.path.basename(fname)
    atoms = atom_types_in_file(fname,1)
    h_yes = 0
    if "H" in str(atoms):
            h_yes = 1
            print('H in '+fname)
            #copy file
            shutil.copy(fname,h_removed_dir+base_fname.split(".pdb")[0]+"_H.pdb")
            #read file with H
            f = open(fname,'r')
            old_contents = f.read().splitlines()
            f.close()
            anumber=1
            new_contents = []
            for ln in old_contents:
                if "H" in ln[12:16] :
                    pass
                else:
                    out_string="{}{:6d}{}".format(ln[0:5],anumber,ln[11:])
                    new_contents.append(out_string)
                    anumber = anumber +1
    if h_yes == 1:
        f = open(fname,"w")
        for ln in new_contents:
            f.write(ln+"\n")
        f.close()
    return h_yes

# Parse the options and setup input parameters
##Warning: prepend and append options have not been tested to see if they work
#on this version of the script
#optparser = OptionParser()

#optparser.add_option('--prepend', default=None, dest='prepend', help='number of closing bases to prepend to the sequence [default: 0]')
#optparser.add_option('--append', default=None, dest='append', help='number of closing bases to append to the sequence [default: 0]')
#optparser.add_option('--overwrite', action="store_true", default=False, dest='overwrite', help='Overwrite existing clipped files if specified, otherwise only missing ones will be created')

#(options, args) = optparser.parse_args()


append = 0
prepend = 0
overwrite = False
prepend = None
append = None
#overwrite = options.overwrite
#assign file name or read from user
#csvfile = args[0]
#csvfile = "Tetraloops_Graham_2.csv"
'''
if options.prepend is not None:
    print('Prepending %s closing bases' % options.prepend)
    prepend = int(options.prepend)
if options.append is not None:
    print('Appending %s closing bases' % options.append)
    append = int(options.append)
'''


data_path = "./"  #path from program to data file
pdb_ext = ".PDB"

pdb_dir = cossmos_work_dir+"/pdbs/" # will saved to shared central folder to avoid duplicate download

#pdb_dir = data_path+"pdbs/" # will saved to shared central folder to avoid duplicate download
clips_dir = data_path+"clips/" # should we use central folder?
#renumbered_dir = data_path+clips_dir+"renumbered\\"
h_removed_dir = clips_dir+"h_removed/"

# Housekeeping... make sure output directories exist, if not create them
check_folder(pdb_dir)
check_folder(clips_dir)
#check_folder(renumbered_dir)
check_folder(h_removed_dir)

#minimum number of lines in pdb file to be valid
min_pdb_lines = 20

# Read the CSV file. This uses the default CoSSMos delimiter
# which is the '#' symbol
data = []
header, ldata = read_csv(csvfile)
data.extend(ldata)

#Create column indices from the csv file header
#Changed the column index names according to new column names 1-24-19
pdbidx = header.index('pdb')
seqidx = header.index('Aseq')
if 'Bseq' in header:
    seqBidx = header.index('Bseq')
asnidx = header.index('Aseq_num')
if 'Bseq_num' in header:
    bsnidx = header.index('Bseq_num')
conidx = header.index('A Residue Conformation 3')
#conidx = header.index('Alooprc3')
#Changed from 'B Residue Conformation 2' to 'Bcbp1' on 9-18-17
if 'Bcbp1' in header:
    bconidx = header.index('Bcbp1')
expidx = header.index('experiment')
residx = header.index('resolution')


#download all pdb files.
pdb_names = []
for i in range(len(data)):
    data_row = data[i]
    pdbid = data_row[pdbidx]

    #if pdbid != '4YBB': # for test
    #   continue


    print("Downloading pdb id: "+pdbid)

    # Check for the full PDB file and download if needed
    # Hack: The file gets downloaded to the current directory and
    #       then moved to the pdbs folder

    # update: 2022/10/16. Many large pdbs do not have pdb file, instead, need use .cif as suffix
    # such as https://files.rcsb.org/download/7S1H.cif
    pdbfile = "%s.pdb" % pdbid
    ciffile = "%s.cif" % pdbid

    pdbname = pdb_dir+pdbfile
    cifname = pdb_dir+ciffile

    # 11/21/2022 check if the pdb file is empty

    if isfile(pdbname):
       if os.stat(pdbname).st_size == 0:
          print('PDB file %s.pdb is empty, redownloading, ' % pdbid)
          os.system('rm '+pdbname)

    if isfile(cifname):
       if os.stat(cifname).st_size == 0:
          print('PDB file %s.pdb is empty, redownloading, ' % pdbid)
          os.system('rm '+cifname)

    # check .pdb download
    if isfile(pdbname) == False and isfile(cifname) == False:
        print('PDB file %s does not exist...downloading' % pdbid)
        url =  "http://www.rcsb.org/pdb/files/%s.pdb" % pdbid
        call(["curl","-L","-O","-s",url])
        if isfile(pdbfile):
            # check if not found, likely only .cif is available on PDB database
            if '404 Not Found' in open(pdbfile).read():
                print('PDB file %s.pdb is not available on PDB website, check .cif' % pdbid)
                os.system('rm '+pdbfile)
            else:
                os.rename(pdbfile,pdbname)
    else:
        print('PDB file %s already exists' % pdbid)

    if isfile(pdbname) == True:
        # check if not found, likely only .cif is available on PDB database
        if '404 Not Found' in open(pdbname).read():
            print('PDB file %s.pdb is not available on PDB website, check .cif' % pdbid)
            os.system('rm '+pdbname)

    if isfile(pdbname) == True:
        # check if pdb file is failed with no atoms inside
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure('asdf',pdbname)
        if len(structure) == 0:
          print('PDB file %s.pdb is invalid, redownloading, ' % pdbid)
          os.system('rm '+pdbname)

    # second round check
    if isfile(pdbname) == False and isfile(cifname) == False:
        print('\nFailed to download PDB file %s, try .cif format' % pdbfile)
        url =  "http://www.rcsb.org/pdb/files/%s.cif" % pdbid
        call(["curl","-L","-O","-s",url])
        if isfile(ciffile):
            os.rename(ciffile,cifname)
    else:
        print('PDB file %s already exists' % pdbid)

    if isfile(pdbname):
        print('PDB file %s already exists' % pdbname)
        pdb_names.append(pdbname)
    elif isfile(cifname):
        print('PDB file %s already exists' % cifname)
        pdb_names.append(cifname)
    else:
        print('Failed to download PDB file %s, double check' % pdbid)


#check all pdb files and be sure there is content.
unique_pdb_names = list(set(pdb_names))
pdbfile_rejects = []
'''
for nm in unique_pdb_names:
    count = len(open(nm).readlines(  ))
    if count < min_pdb_lines:
        pdbfile_rejects.append(nm)
    else:
        print('PDB file %s size OK ' % nm)

print("pdbfile_rejects: ",pdbfile_rejects)
'''

#Create empty lists to be filled during clipping loop
motif_rejects = []
period_rejects = []
#renumbered = []
h_atoms_removed = []

from Bio.PDB.MMCIFParser import MMCIFParser
parser = PDBParser(QUIET=True)
parser_cif = MMCIFParser(QUIET=True)

#Begin clipping loop

for i in range(len(data)):
    data_row = data[i]
    pdbid = data_row[pdbidx]

    #if pdbid != '4YBB': # for test
    #  continue
    print("Loop Processing pdb id: "+pdbid)

    pdbfile = "%s.pdb" % pdbid
    pdbname = '%s/pdbs/%s' % (cossmos_work_dir,pdbfile)

    ciffile = "%s.cif" % pdbid
    cifname = '%s/pdbs/%s' % (cossmos_work_dir,ciffile)

    if (pdbname in pdbfile_rejects) or (cifname in pdbfile_rejects):
        continue

    if isfile(cifname):
      pdbname = cifname
    elif isfile(pdbname):
      pdbname = pdbname
    else:
      print("Failed to find pdb for ",pdbid)
      pdbfile_rejects.append("%s/pdbs/%s.pdb" % (cossmos_work_dir,pdbid))
      pdbfile_rejects.append("%s/pdbs/%s.cif" % (cossmos_work_dir,pdbid))


    # Check for continuous numbering is Bseq_num
    # If continuous numbering add to motif_rejects
    if 'Bseq' in header:
        Aresnum = data_row[asnidx].split()
        Bresnum = data_row[bsnidx].split()
        Anum_list = []
        for res in Aresnum:
            num = res.strip('-UGCA.')
            Anum_list.append(num)
            Bnum_list = []
        for res in Bresnum:
            num = res.strip('-UGCA.')
            Bnum_list.append(num)
        try:
          if int(Anum_list[0]) - int(Bnum_list[0]) == (1 or -1):
              motif_rejects.append(pdbid + '_' + data_row[seqidx] + data_row[seqBidx])
              print("Motif reject %s" % pdbid + '_' + data_row[seqidx] + data_row[seqBidx])
              continue
          if int(Anum_list[-1]) - int(Bnum_list[-1]) == (1 or -1):
              motif_rejects.append(pdbid + '_' + data_row[seqidx] + data_row[seqBidx])
              print("Motif reject %s" % pdbid + '_' + data_row[seqidx] + data_row[seqBidx])
              continue
        except:
              # https://www.rcsb.org/structure/1FJG
              # ['192-U', '191-G', '190L-U', '190K-G', '190J-U']
              # some pdbs have multiple insertions at same nt index. check github issues
              # here we ignore the motif
              motif_rejects.append(pdbid + '_' + data_row[seqidx] + data_row[seqBidx])
              print("Motif reject %s" % pdbid + '_' + data_row[seqidx] + data_row[seqBidx])
              continue

    #Set the chain id
    #B64 G ---- u-turn
    #'1'133 U ---- turn
    #'3'39 C ---- turn
    #AR133 U ---- turn
    #AR2445 A C2p_endo anti

    #chain_id = data_row[conidx].split()[0].rstrip('1234567890').replace("'","")[0]
    # Update here on 2022/10/19: some chain id has two letters, such as 1VVJ chain QA
    #chain_id = data_row[conidx].split()[0].replace("'","")[0]
    #chain_id = data_row[conidx].split()[0].replace("'","")
    if data_row[conidx].split()[0].find('\'')>=0:
      chain_id = str(data_row[conidx].split()[0].split('\'')[1])
    else:
      import re
      #chain_id = 'QA159'
      #chain_id = '0159'
      #chain_id = '13208' # https://www.rcsb.org/fasta/entry/4WSD/display
      chain_id = str(data_row[conidx].split()[0].replace("'",""))
      match = re.match(r"([a-zA-Z]+)([0-9]+)", chain_id, re.I)
      if match:
          items = match.groups()
          chain_id = str(items[0]) #'QA'
      else:
          chain_id = str(chain_id[0])
      #print(items)

    if len(chain_id) == 0:
        chain_id = " "

    #If there is a B chain, set the b chain id
    if 'Bseq' in header:
        #b_chain_id = data_row[bconidx].split()[0].rstrip('1234567890').replace("'", "")[0]
        #b_chain_id = data_row[bconidx].split()[0].replace("'", "")[0]
        # Update here on 2022/10/19: some chain id has two letters, such as 1VVJ chain QA
        #chain_id = data_row[conidx].split()[0].replace("'","")[0]
        #b_chain_id = data_row[bconidx].split()[0].replace("'","")
        import re
        #chain_id = 'QA159'
        #chain_id = '0159'
        if data_row[bconidx].split()[0].find('\'')>=0:
          b_chain_id = str(data_row[bconidx].split()[0].split('\'')[1])
        else:
          #b_chain_id = data_row[bconidx].split()[0].replace("'", "")[0]
          b_chain_id = str(data_row[bconidx].split()[0].replace("'", ""))
          match = re.match(r"([a-zA-Z]+)([0-9]+)", b_chain_id, re.I)
          if match:
              items = match.groups()
              b_chain_id = str(items[0]) #'QA'
          else:
              b_chain_id = str(b_chain_id[0])

        if len(b_chain_id) == 0:
            b_chain_id = " "
        #print b_chain_id

    #Set the first base equal to the first base in the A loop
    first_base = data_row[conidx].split()[0]

    #This reverses the order of the B sequence 7-6-18
    if 'Bseq' in header:
        bseq = data_row[seqBidx]
        bseq = bseq[::-1]
        bseq = bseq[1:-1]

    #Set the filename equal to the pdbid_sequence_firstbase
    #print("header: ",header)
    if 'Bseq' in header:
        #Edited on 3-4-19
        #basename = "%s_%s_%s" % (pdbid,data_row[seqidx][1:-1] + data_row[seqBidx][1:-1],first_base)
        basename = "%s_%s_%s" % (pdbid,data_row[seqidx][1:-1] + bseq, first_base)
    else:
        #Edited on 1-29-19
        basename = "%s_%s_%s" % (pdbid, data_row[seqidx][1:-1], first_base)

    '''
    print("basename: ",basename)
    print("seqidx: ",seqidx)
    print("data_row: ",data_row)
    print("data_row[seqidx]: ",data_row[seqidx])
    print("data_row[seqidx][1:-1]: ",data_row[seqidx][1:-1])
    print("bseq: ",bseq)
    print("first_base: ",first_base)
    '''

    #Add the .pdb extension to the filename and direct it to the clips folder
    clipname = clips_dir+basename+pdb_ext

    ## 2022/10/19: Add file check, if already generated, skip
    if os.path.exists(clipname):
        # 11/21/2022 check if the pdb file is empty
       if os.stat(clipname).st_size == 0:
          print('clip file %s is empty, reprocessing, ' % clipname)
          os.system('rm '+clipname)
       else:
          print(clipname, ' has been generated! Skip!')
          continue

    if os.path.exists(clipname):
       print(clipname, ' has been generated! Skip!')
       continue

    #Get the chain from the first model in the pdb file
    try:
      if '.cif' in pdbname:
        structure = parser_cif.get_structure('asdf',pdbname)
      else:
        structure = parser.get_structure('asdf',pdbname)
    except:
        print("Failed to read ",pdbname)
        continue

    # for NMR, model id is not always 1
    model_id = int(data_row[header.index('model')].replace('*',''))-1
    print("model_id: ",model_id)
    print("structure: ",structure, len(structure))
    for model in structure:
      print("Model ID:", model.id)

    model = structure[model_id]
    for chain in model:
        chain.id = str(chain.get_id()) # force chain id to string, i.e. '1'

    chain = model[chain_id]
    old_name = chain.get_id()

    if 'Bseq' in header:
        b_chain = model[b_chain_id]
        old_name_bchain =  b_chain.get_id()



    if len(chain_id) > 1:
      # 2022/10/19: current biopython doesn't support chain name with more than two letters, rename it to one-letter here
      #for model in structure:
      #    for chain in model:
      #        old_name = chain.get_id()
      #        if old_name == chain_id:
      #          print("old_name: ",old_name)
      #          print("chain_id: ",chain_id)
      #          new_name = '?' # hard to find single letter to fit all possible pdb cases, so try ? first, may need fix later
      #          chain.id = new_name
      #          chain_id = new_name
      #          print('Changing chain name from ',old_name, ' to ', new_name)
      old_name = chain.get_id()

      print("old_name: ",old_name)
      print("chain_id: ",chain_id)
      new_name = '?' # hard to find single letter to fit all possible pdb cases, so try ? first, may need fix later
      chain.id = new_name
      chain_id = new_name
      print('Changing chain name from ',old_name, ' to ', new_name)

    # add on Mar 30, 2023 to avoid long chain id in b_chain_id
    if 'Bseq' in header:
      if len(b_chain_id) > 1:

        if old_name == old_name_bchain: # if the two chain ids are same, use the short name '?'

          print("old_name_bchain: ",old_name_bchain)
          print("b_chain_id: ",b_chain_id)
          new_name = '?' # hard to find single letter to fit all possible pdb cases, so try ? first, may need fix later
          b_chain.id = new_name
          b_chain_id = new_name
          print('Changing b chain name from ',old_name_bchain, ' to ', new_name)
        else:
          print("old_name_bchain: ",old_name_bchain)
          print("b_chain_id: ",b_chain_id)
          new_name = '!' # hard to find single letter to fit all possible pdb cases, so try ? first, may need fix later
          b_chain.id = new_name
          b_chain_id = new_name
          print('Changing b chain name from ',old_name_bchain, ' to ', new_name)


    #This reverses the order of the B residues 7-6-18
    if 'Bseq' in header:
        bresids = data_row[bsnidx]
        bresids_split = bresids.split()
        bresids_sorted = bresids_split[::-1]
        bresids = ' '.join(bresids_sorted)

    #If there is a bchain, grab the A and B residue ids and clean them up
    if 'Bseq' in header:

        resids = data_row[asnidx]
        #bresids = data_row[bsnidx]
        if '.' in bresids:
            print('Contains period.  Deleting periods in %s' % clipname)
            period_rejects.append(clipname)
            print (bresids)
            bresids = bresids.replace('.', '')
            print (bresids)
        if bresids[0] == '-':
            print ('Negative resid')
            bresids_split = bresids.split()
            bresids2 = []
            bposresids = []
            for r in bresids_split:
                r2 = r.rsplit('-')
                bposresids.append(r2[1])
            for r in bposresids:
                r2 = "-" + r
                bresids2.append(r2)
        else:
            bresids_split = bresids.split()
            bresids2 = []
            for r in bresids_split:
                r2 = r.rsplit('-')
                bresids2.append(r2[0])
        bresids2 = bresids2[1:-1]


    #If only A chain, grab the residues and clean them up
    else:
        resids = data_row[asnidx]


    if '.' in resids:
        print('Contains period.  Deleting periods in %s' % clipname)
        period_rejects.append(clipname)
        print (resids)
        resids = resids.replace('.','')
        print (resids)

    if resids[0] == '-':
        print ('Negative resid')
        resids_split = resids.split()
        resids2 = []
        posresids = []
        for r in resids_split:
            r2 = r.rsplit('-')
            posresids.append(r2[1])
        for r in posresids:
            r2 = "-" + r
            resids2.append(r2)

    else:
        resids_split = resids.split()
        resids2 = []
        for r in resids_split:
            r2 = r.rsplit('-')
            resids2.append(r2[0])

    print('Processing sequence: %s' % basename)

    # Determine and adjust the ranges based on whether or not
    # padding is being done
    #beg = int(resids.split()[0].rsplit('-',1)[0])
    #end = int(resids.split()[-1].rsplit('-',1)[0])


    # Determine and adjust the ranges based on whether or not
    # padding is being done
    #print("resids2: ", resids2)
    #print("bresids2: ", bresids2)


    #Added line to remove second closing base pair residues 1-29-19
    resids2 = resids2[1:-1]

    #Set the range to select from the pdb file
    # RangeSelect  and BRangeSelect have been modified to renumber the residues
    # matching the selection criteria, starting at 1

    #If there is a B chain, but it is the same chain as the A chain
    #Use the RangeSelect function
    if 'Bseq' in header:
        print("This SSE has two strands")
        #if chain_id == b_chain_id:
        print('old_name: ', old_name)
        print('chain_id: ', chain_id)
        print('b_chain_id: ', b_chain_id, type(b_chain_id))
        if chain_id == b_chain_id:
            resids2 = resids2 + bresids2
            positions_resids2 = zip(range(1, len(resids2) + 1), resids2)
            positions_resids2_dict = dict(positions_resids2)
            rselect = RangeSelect(chain_id, resids2, positions_resids2_dict)

        #If there is a B chain, but it is different from the A chain
        #Use the BRangeSelect function
        else:
            total_resids2 = resids2 + bresids2
            positions_resids2 = zip(range(1, len(total_resids2) + 1), total_resids2)
            print (positions_resids2)
            positions_resids2_dict = dict(positions_resids2)
            rselect = BRangeSelect(chain_id, b_chain_id, resids2, bresids2, positions_resids2_dict)

    #If there is only an A chain, use the RangeSelect function
    else:
        print("inside 1", resids2, chain_id)
        positions_resids2 = zip(range(1, len(resids2) + 1), resids2)
        positions_resids2_dict = dict(positions_resids2)
        rselect = RangeSelect(chain_id, resids2, positions_resids2_dict)

    # Save out the PDB file with modified residue numbers


    ## Add on Mar 30, 2023 to avoid  long chain ids
    if 'Bseq' in header:
      chains_to_keep = [chain_id, b_chain_id]
    else:
      chains_to_keep = [chain_id]

    # Create a new structure object
    from Bio.PDB.Model import Model
    from Bio.PDB.Structure import Structure
    from Bio.PDB.Chain import Chain

    #try: # commented this on April 2, 2023
    #    io = PDBIO()
    #    io.set_structure(structure)
    #    #io.set_structure(chain)
    #    print("rselect: ",rselect)
    #    print("clipname: ",clipname)
    #    print("chain_id: ",chain_id)
    #    print("rselect.resids2: ",rselect.resids2)
    #    io.save(clipname,rselect)
    #except:

    io = PDBIO()
    #print("Switch to multi-letter chain mode for pdb writing")
    # Create a new model object
    filtered_chains = Model("merged")
    # Create a new structure object with the merged model
    #filtered_structure = Structure("merged")
    for chain in model:
      #if len(chain.get_id()) > 1 and chain.get_id() not in chains_to_keep:
      #  print("removing ", chain.get_id())
      #  #chain.id = '*' # temporary change multi-letter chain id to single letter for model saving
      #  #chain.detach_parent()
      #  continue
      #else:
      #  filtered_chains.add(Chain(chain.get_id(), chain.get_unpacked_list()))
      if len(chain.get_id()) > 1:
        continue
      elif str(chain.get_id()) not in chains_to_keep:
        continue
      else:
        chain.id = str(chain.get_id())
        print("adding ", chain.get_id())
        filtered_chains.add(chain)

    #filtered_structure.add(filtered_chains)
    #io.set_structure(structure)
    io.set_structure(filtered_chains)
    #io.set_structure(chain)
    #print("rselect: ",rselect)
    #print("clipname: ",clipname)
    #print("chain_id: ",chain_id, type(chain_id))
    #print("rselect.resids2: ",rselect.resids2)
    io.save(clipname,rselect)
    #determine if numbers are in increments of 1 or not.
    #If not, renumber in file and copy original to
    #renumbered directory
    #if in_order(resids, beg, end) == 0:
        #renumber_resids(clipname,renumbered_dir)
        #print(clipname+" renumbered")
        #renumbered.append(clipname)
    #else:
        #print(clipname+" numbers ok")

    removed = purge_h_atoms(clipname,h_removed_dir)
    if removed == 1:
        print ("H-atoms removed from "+clipname)
        h_atoms_removed.append(clipname)
    else:
        print ("no H-atoms in "+clipname)

print('All done processing CSV file: %s' % csvfile)
#print changes and info to files to clips dir
write_rejects(clips_dir+"pdbfile_rejects.txt",pdbfile_rejects)
write_rejects(clips_dir+"periods_removed.txt",period_rejects)
#write_rejects(clips_dir+"renumbered.txt",renumbered)
write_rejects(clips_dir+"H_atoms_removed.txt",h_atoms_removed)
write_rejects(clips_dir+"motif_rejects.txt",motif_rejects)


In [ ]:
#Check files

# Not empty
# Not missing atoms
#  Does not have multiple coordinates for the same atom
#


# -*- coding: utf-8 -*-
"""
Created on Wed Oct 12 09:17:09 2016
Modified on Thur Oct 20 11:37:05 2016 by kerichardson

@author: kirkpacc
"""

from os import listdir
from subprocess import call

import shutil
import os.path
from os.path import isfile, join, basename, isdir


import numpy as np
import numexpr as ne
import csv

from itertools import combinations, groupby
from operator import itemgetter

from Bio.PDB import *
from optparse import OptionParser



#finds files in directory that are missing bases
#assumes P, P01, P02 followed by nine primes (sugar)
#and then presence or absence of bases.  If total in
#the section is >12, then assumed bases are present
#KR Also checks for multiple atom coordinates and
#KR files that only contain one line

def rough_check(g,content):
    #returns 0/1 for count is not/ok for group(g) in content
    count = 0
    for line in content:
        if (line[0:4] == "ATOM") and (int(line[24:27]) == g):
            count = count + 1
    if (count > 12) :
        result = 1
    else:
        result = 0
    return result

def good_check(g,content,fname):
    #returns 0/1 for count is not/ok for group(g) in content
    count = 0
    review = []
    mult_atom = 0
    #KR Check for files with only one line
    if len(content) <= 1:
        is_empty = 1
        result = 0
        return result, is_empty
    else:
        is_empty = 0
    for line in content:
        #KR Check for multiple atom coordinates
        if ("A" or "B") in line[16:17]:
            mult_atom = mult_atom + 1

        resid = line[24:27]
        resid = re.sub('[^A-Za-z0-9]+', '', resid) # added on 2022/10/18, remove escape characters

        if (line[0:4] == "ATOM" or line[0:6] == 'HETATM' or line[0:5] == 'HETAT') and (int(resid) == g):
            review.append(line)
            count = count + 1
    #check first three lines for P
    #must check that values being tested are within range
    p_ok = 0
    tick_ok = 0
    if count > 12:
        for i in range(0,3):
            if "P" in review[i][:18]:
                p_ok = p_ok + 1
        #check next nine lines for '
        for i in range(3,12):
            if "'" in review[i]:
                tick_ok = tick_ok +1
                #print(tick_ok)

    if ((count > 12) and (p_ok == 3) and (tick_ok == 9)and (mult_atom == 0) and (is_empty == 0)):
       result = 1
    else:
       result = 0
    return result,mult_atom,p_ok,tick_ok,count, is_empty

#KR Check for folder in path...if it does not exist, create it
def check_folder(folder):
    if os.path.isdir(folder) == False:
        print('Folder \'%s\' does not exist...creating' % folder)
        try:
            os.makedirs(folder, exist_ok=True)
        except OSError as exception:
            if exception.errno != errno.EEXIST:
                raise
        if not os.path.isdir(folder):
          print("Failed to create folder ", folder)



import glob
import os
import re
import shutil
import csv

script_path = "./"
clip_path = script_path+"clips/" #relative path to clip directory
check_folder(script_path+'good_clips')
check_folder(script_path+'bad_clips')
good_files = script_path+"good_clips/" #qssumes dir exists
bad_files = script_path+"bad_clips/" #assumes dir exists
#sequence = '*.pdb'
sequence = '*.PDB'
#sequence = '1MSY_CGUAAG_A.PDB'

#file_list = glob.glob(pdb_path)
clips_pdb = clip_path+'%s'%sequence
file_list = glob.glob(clips_pdb)
print("file_list: ",file_list, ' from ',clips_pdb)
#loop through all files and test
split_line = []
bad_results = []
for fname in file_list:
    print("Checking "+fname)
    path,base = os.path.split(fname)
    file_ok = 1
    with open(fname,'r', encoding='utf-8') as f:
        content = f.readlines()
        '''
        content=[]
        for line in content_tmp:
            print('befor: ',line)
            line = re.sub('[^A-Za-z0-9]+', '', line) # added on 2022/10/18, remove escape characters

            content.append(line) # added on 2022/10/18, remove escape characters
            print('after: ',line)
        '''
        if len(content) <= 1:
            is_empty = 1
            file_ok = 0
            bad_results.append([fname,0,0,0,0,0,0,1])
        else:
            is_empty = 0
            resid = content[len(content)-3][24:27].strip()
            resid = re.sub('[^A-Za-z0-9]+', '', resid) # added on 2022/10/18, remove escape characters
            #if (content[len(content)-3][24:27].strip()).isdigit():
            if resid.isdigit():
                resids = []
                for line in content:
                    resid = line[24:27]
                    resid = re.sub('[^A-Za-z0-9]+', '', resid) # added on 2022/10/18, remove escape characters
                    resids.append(resid)
                max_groups = int(max(resids))
                for g in range(1,max_groups+1):
                    group_ok,mult_atom,p_ok,tick_ok,count,is_empty = good_check(g,content,fname)
                    if group_ok == 0:
                        file_ok = 0
                        bad_results.append([fname,g,group_ok,p_ok,tick_ok,count,mult_atom,is_empty])
            else:
                max_groups = 0
                file_ok = 0
                bad_results.append([fname,0,0,0,0,0,0,0])
    if file_ok == 1: #write to good_files directory
        shutil.copy(fname,good_files+base)
        print("file ok. copy to good_clips "+fname)
    else:  #write to bad_files directgory
        shutil.copy(fname,bad_files+base)
        print("file not ok. copy to bad_clips "+fname)

#write badfile results to directory
with open(bad_files+"bad_file_data.txt", "w") as csv_file:
    writer = csv.writer(csv_file, delimiter='\t')
    writer.writerow(["File","group","file_ok","P Count","Sugar Count","Total Count","Total Mult Atom Count","IsEmpty"])
    for line in bad_results:
        writer.writerow(line)



Streaming output truncated to the last 5000 lines.
Checking ./clips/4V9O_GCUGUUCGC_GA2552_1.PDB
file ok. copy to good_clips ./clips/4V9O_GCUGUUCGC_GA2552_1.PDB
Checking ./clips/4V9O_CGUAAUCCG_GA2210_1.PDB
file ok. copy to good_clips ./clips/4V9O_CGUAAUCCG_GA2210_1.PDB
Checking ./clips/4V9O_GGUAAGUUC_GA1951_1.PDB
file ok. copy to good_clips ./clips/4V9O_GGUAAGUUC_GA1951_1.PDB
Checking ./clips/4V9O_UAGAGAAUA_GA1631_1.PDB
file ok. copy to good_clips ./clips/4V9O_UAGAGAAUA_GA1631_1.PDB
Checking ./clips/4V9O_GGCAAAUCC_GA1493_1.PDB
file ok. copy to good_clips ./clips/4V9O_GGCAAAUCC_GA1493_1.PDB
Checking ./clips/4V9O_CGUUAAUCG_GA1325_1.PDB
file ok. copy to good_clips ./clips/4V9O_CGUUAAUCG_GA1325_1.PDB
Checking ./clips/4V9O_GUGAAAGGC_GA780_1.PDB
file ok. copy to good_clips ./clips/4V9O_GUGAAAGGC_GA780_1.PDB
Checking ./clips/4V9O_CUUAACUGG_GA642_1.PDB
file ok. copy to good_clips ./clips/4V9O_CUUAACUGG_GA642_1.PDB
Checking ./clips/4V9O_CCGAAUAGG_GA612_1.PDB
file ok. copy to good_clips ./clips/4

In [ ]:
# Reorder Residues

#After running this script, each clip in the “good_clips” folder should start with a residue id of 1 (column 6 of the .pdb). It will also create an “original” folder containing all of the clips before they were reordered.


#!/usr/bin/env python

import glob

from os import listdir
from subprocess import call
import re
import shutil
import os.path
from os.path import isfile, join, basename, isdir


import numpy as np
import numexpr as ne

from itertools import combinations, groupby
from operator import itemgetter

from Bio.PDB import *
from optparse import OptionParser

def read_csv(file):
    csvfile = open(file, 'r')

    # replace csv library processing
    # if list of files was opened with excel it is
    # possible that some lines starte/stop with "
    # and then contents are not split correctly.
    lines = [line.strip() for line in csvfile]
    csvfile.close()

    header = lines[0].strip('"').split('#')
    data = []
    for l in lines[1:]:
        data.append(l.strip('"').split('#'))

    return header, data


def check_folder(folder):
    if os.path.isdir(folder) == False:
        print('Folder \'%s\' does not exist...creating' % folder)
        try:
            os.makedirs(folder, exist_ok=True)
        except OSError as exception:
            if exception.errno != errno.EEXIST:
                raise
        if not os.path.isdir(folder):
          print("Failed to create folder ", folder)


def reorder_resids(clipname):
    base_fname=os.path.basename(clipname)
    shutil.copy(clipname,reordered_dir+"/"+base_fname)

    #f = open(clipname, 'r')
    #lines = f.readlines()

    # Open the file, modied on 03/22/2024 to remove \x01-\x09 in each pdb
    with open(clipname, 'r') as f:
        # Read lines and remove special characters using regular expression
        lines = [re.sub(r'[\x01-\x09]', ' ', line) for line in f]

    split_lines = []
    # added on 2022/10/18, remove escape characters
    for l in lines:
        if re.sub('[^A-Za-z0-9]+', '', l[0:3]) == 'END':
            lines.remove(l)
    for l in lines:
        if re.sub('[^A-Za-z0-9]+', '', l[0:3]) == 'TER':
            lines.remove(l)
    for l in lines:
        if re.sub('[^A-Za-z0-9]+', '', l[0:5]) == 'MODEL':
            lines.remove(l)
    for l in lines:
        if re.sub('[^A-Za-z0-9]+', '', l[0:3]) == 'END':
            lines.remove(l)

    try:
      sorted_lines = sorted(lines, key=lambda x: int(x.split()[5]))
    except:
      # added on 2022/10/18, remove escape characters
      #print(int(re.sub('[^A-Za-z0-9]+', ' ', lines[0]).split()[5]))
      '''
      for lin in lines:
        if len(re.sub('[^A-Za-z0-9]+', ' ', lin).split())<6:
          print('error:',lin,clipname)
          aa = re.sub('[^A-Za-z0-9]+', '', lin[0:3])
          print("check: ",aa, len(aa), type(aa))
      '''
      # added on 2023/02/23, add ? ! in regex for keeping chain id ? for those pdb has length id has more than 1
      sorted_lines = sorted(lines, key=lambda x: int(re.sub('[^A-Za-z0-9?!]+', ' ', x).split()[5]))

    f = open(clipname, "w")
    for line in sorted_lines:
        f.write(line)
    f.write("TER" + "\n")
    f.write("END")
    f.close()

#import sys
#print(sys.argv, len(sys.argv))

#if len(sys.argv) != 2:
#    print('The parameter is not correct!')
#    exit(-1)


#process_dir = sys.argv[1]

process_dir = 'good_clips/'

script_path = "./"
reordered_dir = script_path+"original/"

check_folder(reordered_dir)

#sequence = '*.pdb'
sequence = process_dir+'/*.PDB'

file_list = glob.glob(script_path+'%s'%sequence)

#print("file_list: ",file_list)
for fname in file_list:
    reorder_resids(fname)

In [ ]:
#Define DSSR Characterization Functions

#DSSR (or Dissecting the Spatial Structures of RNA) is the software we use for annotating RNA 3D files. More information about DSSR can be found towards the end of this guide.

# -*- coding: utf-8 -*-
"""
Created on Wed Mar  8 20:28:38 2017

@author: kirkpacc
"""
from __future__ import division
from operator import itemgetter
import csv
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
from collections import Counter
import itertools
from tabulate import tabulate
import os
import re
import datetime

import json
import glob
import os
import subprocess
import sys
import re


from tkinter import Tk

def dssr_characterization(process_dir, blacklist):


    #%shell chmod 755 /content/drive/MyDrive/AML/AML_Project/


    def clean_json (fname):
        f1 = open(fname, 'r')
        f2 = open(fname+".tmp", 'w')
        for line in f1:
            f2.write(line.replace('\\', '\\\\'))
        f1.close()
        f2.close()
        os.remove(fname)
        os.rename(fname+".tmp",fname)

    def find_it (look_here,ind1,fld1,ind2,fld2):
        'finds value'
        if (fld1 in look_here[ind1]) and (fld2 in look_here[ind1][fld1][ind2]) :
            return str(look_here[ind1][fld1][ind2][fld2])
        else:
            return "blank"

    def dict_count (dict,elem):
        'returns num of occurances of elem in dict'
        if elem in dict:
            return  range(len(dict[elem]))
        else:
            return range(0,0);

    '''
    #determine if version 2 or 3
    if sys.version_info < (3,0,0):
        pversion = 2
        import tkinter, filedialog
        from tkinter import *
    else:
        pversion = 3
        from tkinter import filedialog
        from tkinter import *
    '''
    import sys

    '''
    print(sys.argv, len(sys.argv))

    if len(sys.argv) != 3:
        print('The parameter is not correct!')
        exit(-1)


    DSSR_run_path = sys.argv[1]
    DSSR_data_path = sys.argv[2]
    '''

    #DSSR_run_path = '/content/drive/MyDrive/AML/AML_Project/'+'/x3dna-dssr'
    # 2023/10/29: update to v2.3.2
    DSSR_run_path = '/content/drive/MyDrive/AML/AML_Project/x3dna-dssr_v2.3.2'
    %shell chmod 755 /content/drive/MyDrive/AML/AML_Project/x3dna-dssr_v2.3.2
    %shell /content/drive/MyDrive/AML/AML_Project/x3dna-dssr_v2.3.2 -v


    #process_dir = cur_work_dir +'/good_clips/'
    print("Entering ",process_dir)
    if not os.path.exists(process_dir):
      print("Warning: Failed to find directory: ",process_dir)
      exit(-1)

    os.chdir(process_dir)


    try:
        family_dirs = [ f.path for f in os.scandir(process_dir) if f.is_dir() ]

        for family_dir in family_dirs:
            if family_dir.split('/')[-1].strip() in blacklist:
              print("!!!! Skipping folder ", family_dir)
              continue
            print("!!!！ Processing family: ",family_dir)
            os.chdir(family_dir)

            #root = Tk()
            #root.withdraw()
            #root.deiconify()
            #root.destroy()
            #root.fileName = filedialog.askopenfilename(filetypes=(("text","*.txt"),("All","*.*")))
            #print(root.fileName)

            #path to DSSR executable
            '''
            if pversion == 2:
                #root.fileName = filedialog.askopenfilename(title="Select x3dna-dssr.exe",filetypes=(("exe","*.exe"),("All","*.*")))
                root.fileName = filedialog.askopenfilename(title="Select x3dna-dssr.exe",filetypes=(("exe","*.exe"),("All","*.*")))
                print(root.fileName)
            else:
                #root.fileName = filedialog.askopenfilename(title="Select x3dna-dssr.exe",filetypes=(("exe","*.exe"),("All","*.*")))
                root.fileName = filedialog.askopenfilename(title="Select x3dna-dssr.exe",filetypes=(("exe","*.exe"),("All","*.*")))
                print(root.fileName)
            DSSR_run_path = root.fileName
            '''
            #DSSR_run_path = './x3dna-dssr'

            #path to dssr input/output files
            #output to same directorry as PDB files
            '''
            if pversion == 2:
                root.dirName = filedialog.askdirectory(title="Choose directory of PDB files")
            else:
                root.dirName = filedialog.askdirectory(title="Choose directory of PDB files")
            #print root.dirName
            #DSSR_data_path = ".\\"
            '''

            #DSSR_data_path = root.dirName+"/"

            #print(DSSR_data_path)
            all_files = glob.glob("./*")
            pdb_files = []
            for f in all_files:
              if '.PDB' in f:
                  pdb_files.append(f);

            #run DSSR for each PDB file
            for file in pdb_files:
              basefname = file.replace(".PDB","")

              # Nov 8, added to skip pdb whose dssr results are available
              try:
                with open(basefname+".json", 'r') as infile:
                    json.load(infile, strict=False)
                continue
              except:
                args = " --input=\""+file+"\" --json -o=\""+basefname+".json\" --prefix=temp --more --non-pair --u-turn --po4 --idstr=long"
                run_string = DSSR_run_path+args
                #subprocess.call(run_string)
                #print(run_string)
                os.system(run_string)
                #json file needs to be cleaned of single escape characters
                clean_json (basefname+".json")

            #delete merged.json if it exists
            if os.path.isfile("./merged.json"):
              os.remove("./merged.json")
              #print("File Removed!")

            #list files
            jsons = glob.glob('./*.json')
            #print (jsons)

            #loop through files--append contents and ,
            f = open("./merged.json", "w")
            f.write("[\n")
            for fname in jsons:
              fr = open(fname)
              f.write(fr.read())
              if (fname != jsons[-1]):
                  f.write(",\n")
            f.write("]")
            f.close()

            #delete all files with form temp-*
            for f in glob.glob("./temp-*"):
              os.remove(f)

            # code for json_bp.py


            #from tkinter import filedialog
            #from tkinter import *

            #output file name
            fname="./bp.txt"
            #following outputs all data to file when =1,
            #smaller output when =0
            out_more=0

            #output to screen (=1) or silent (0)
            out_screen=0

            #field delimiter and eol
            delim="#"
            eol="\n"

            #open json file
            json_fname = "./merged.json"
            clean_json(json_fname)
            with open(json_fname) as json_file:
              json_data = json.load(json_file, strict=False)


            #names for column labels. should end with eol character
            #User must be sure this list is consistent with
            #value of out_more
            labels = ['sequence','nt1','nt2','bp','name']

            #open output file
            fp = open(fname, "w")
            #write dolumn labels
            for l in labels:
              fp.write(l+delim)
            fp.write(eol)


            results = []
            for ind in range(len(json_data)):
                  try:
                    f1=json_data[ind]['metadata']['str_id']+delim
                  except:
                    print("Error: ",json_data[ind]) # no nucleotides found
                    continue

                  for ind2 in dict_count(json_data[ind],'pairs'):
                      f2=find_it(json_data,ind,'pairs',ind2,'nt1')
                      f2 = re.sub("[^a-z0-9?.]+","", f2, flags=re.IGNORECASE) # 03/13/2023: ? for those chain id longer than 1 letter
                      f2s=f2[len(f2)-2:len(f2)-1]+delim
                      f2l=f2+delim+f2s

                      f3=find_it(json_data,ind,'pairs',ind2,'nt2')
                      f3 = re.sub("[^a-z0-9?.]+","", f3, flags=re.IGNORECASE) # 03/13/2023: ? for those chain id longer than 1 letter
                      f3s=f3[len(f3)-2:len(f3)-1]+delim
                      f3l=f3+delim+f3s


                      f4=find_it(json_data,ind,'pairs',ind2,'bp')
                      f4s=f4[len(f4)-1:len(f4)-1]+delim
                      f4l=f4+f4s

                      f5=find_it(json_data,ind,'pairs',ind2,'name')


                      if out_more == 1:
                          fp.write(f1+f2l+f3l+f4l+f5+eol)
                          if out_screen == 1:
                              print(f1+f2l+f3l+f4l+f5)
                      else:
                          fp.write(f1+f2s+f3s+f4l+f5+eol)
                          if out_screen == 1:
                              print(f1+f2s+f3s+f4l+f5)

            fp.close()

            #code for json_stacks.py

            def find_in_str(str):
              #returns the single occurence of a string in list that is in str
              #exits if none of more than one string in list is found in str
              #list is given below
              to_look_for=['<<','<>','><','>>']
              x=[int(str.find(to_look_for[0])),int(str.find(to_look_for[1])),int(str.find(to_look_for[2])),int(str.find(to_look_for[3]))]
              if x == [-1,-1,-1,-1]:
                  #not found
                  return "";
              else:
                  return to_look_for[x.index(max(x))]




              #if max((x))-3 == sum((x)) :
                  #pos=x.index(max(x));
                  #return to_look_for[pos]
              #else:
                  #print ('not ok in find_in_str')
                  #exit()

            #output file name
            print("Processing ./stacks.txt")
            fname="./stacks.txt"
            #following outputs all data to file when =1,
            #smaller output when =0
            out_more=0

            #output to screen (=1) or silent (0)
            out_screen=0

            #field delimiter and eol
            delim="#"
            eol="\n"


            #json_fname = '.\\merged.json'
            json_fname = "./merged.json"
            clean_json(json_fname)
            with open(json_fname) as json_file:
              json_data = json.load(json_file, strict=False)

            #names for column labels. should end with eol character
            #User must be sure this list is consistent with
            #value of out_more
            labels = ['sequence','nt1','nt2','stacking']


            #open output file
            fp = open(fname, "w")
            #write dolumn labels
            for l in labels:
              fp.write(l+delim)
            fp.write(eol)

            #begin program
            for ind in range(len(json_data)):
              #find sequence
              try:
                f1=json_data[ind]['metadata']['str_id']+delim
              except:
                print("Error: ",json_data[ind]) # no nucleotides found
                continue

              #determine number of nonPairs. Loop to find nt1,nt2,stacking
              for ind2 in dict_count(json_data[ind],'nonPairs'):

                  #f2 is nt1
                  f2=find_it(json_data,ind,'nonPairs',ind2,'nt1')
                  f2 = re.sub("[^a-z0-9?.]+","", f2, flags=re.IGNORECASE) # 03/13/2023: ? for those chain id longer than 1 letter:
                  f2s = f2[len(f2)-2:len(f2)-1]+delim
                  f2l=f2+f2[len(f2)-2:len(f2)-1]+delim

                  #f3 is nt2
                  f3=find_it(json_data,ind,'nonPairs',ind2,'nt2')
                  f3 = re.sub("[^a-z0-9?.]+","", f3, flags=re.IGNORECASE) # 03/13/2023: ? for those chain id longer than 1 letter:
                  f3s = f3[len(f3)-2:len(f3)-1]+delim
                  f3l=f3+f3[len(f3)-2:len(f3)-1]+delim

                  #f4 is stacking
                  f4=find_it(json_data,ind,'nonPairs',ind2,'stacking')+delim
                  f4s = find_in_str(f4)
                  f4l = f4+ find_in_str(f4) +delim

                  if f4s == '':
                    # 2023/10/28 ignore the hbonding pairs
                    continue

                  outstringl=f1+f2+f3+f4

                  #output to file only if all fields are not blank

                  if out_more == 1:
                      fp.write(f1+f2l+f3l+f4l+eol)
                      if out_screen == 1:
                          print(f1+f2l+f3l+f4l)
                  else:
                      fp.write(f1+f2s+f3s+f4s+eol)
                      if out_screen == 1:
                          print(f1+f2s+f3s+f4s)

            fp.close()

            #code for json_hbond

            #output file name
            print("Processing ./hbonds.txt")
            fname="./hbonds.txt"
            #following outputs all data to file when =1,
            #smaller output when =0
            out_more=0

            #output to screen (=1) or silent (0)
            out_screen=0

            #field delimiter and eol
            delim="#"
            eol="\n"

            #open json file
            json_fname = "./merged.json"
            clean_json(json_fname)
            with open(json_fname) as json_file:
              json_data = json.load(json_file, strict=False)

            #names for column labels. should end with eol character
            #User must be sure this list is consistent with
            #value of out_more
            labels = ['sequence','nt1','nt2','hbond1','hbond2','hbond3', 'hbond4']

            #open output file
            fp = open(fname, "w")
            #write dolumn labels
            for l in labels:
              fp.write(l+delim)
            fp.write(eol)


            results = []
            for ind in range(len(json_data)):
                  try:
                    f1=json_data[ind]['metadata']['str_id']+delim
                  except:
                    print("Error: ",json_data[ind]) # no nucleotides found
                    continue

                  for ind2 in dict_count(json_data[ind],'nonPairs'):
                      f2=find_it(json_data,ind,'nonPairs',ind2,'nt1')
                      f2 = re.sub("[^a-z0-9?.]+","", f2, flags=re.IGNORECASE) # 03/13/2023: ? for those chain id longer than 1 letter:
                      f2s=f2[len(f2)-2:len(f2)-1]+delim
                      f2l=f2+delim+f2s

                      f3=find_it(json_data,ind,'nonPairs',ind2,'nt2')

                      f3 = re.sub("[^a-z0-9?.]+","", f3, flags=re.IGNORECASE) # 03/13/2023: ? for those chain id longer than 1 letter:
                      f3s=f3[len(f3)-2:len(f3)-1]+delim
                      f3l=f3+delim+f3s

                      f4=find_it(json_data,ind,'nonPairs',ind2,'hbonds_desc')+delim
                      #remove [  ]
                      f4=re.sub(r'\[.*?\]','', f4)
                      #break at comma into two strings
                      f4=re.sub(',', delim, f4)

                      if out_more == 1:
                          fp.write(f1+f2l+f3l+f4+eol)
                          if out_screen == 1:
                              print(f1+f2l+f3l+f4)

                      else:
                          fp.write(f1+f2s+f3s+f4+eol)
                          if out_screen == 1:
                              print(f1+f2s+f3s+f4)



            fp.close()

            #code for json_conf_pucker.py

            #output file name
            print("Processing ./conf_pucker.txt")
            fname="./conf_pucker.txt"
            #following outputs all data to file when =1,
            #smaller output when =0
            out_more=0

            #output to screen (=1) or silent (0)
            out_screen=0

            #field delimiter and eol
            delim="#"
            eol="\n"


            #open json file
            json_fname = "./merged.json"
            with open(json_fname) as json_file:
              json_data = json.load(json_file, strict=False)


            #names for column labels. should end with eol character
            #User must be sure this list is consistent with
            #value of out_more
            labels = ['sequence','nt','base_conf','puckering']

            #open output file
            fp = open(fname, "w")
            #write dolumn labels
            for l in labels:
              fp.write(l+delim)
            fp.write(eol)
            #begin program
            for ind in range(len(json_data)):
              #find sequence
              try:
                f1=json_data[ind]['metadata']['str_id']+delim
              except:
                  print("Error: ",json_data[ind]) # no nucleotides found
                  continue

              #determine number of nts and start loop
              for ind2 in dict_count(json_data[ind],'nts'):

                  #f2 is index
                  f2=find_it(json_data,ind,'nts',ind2,'index')+delim

                  #f3 is baseSugar_conf
                  f3=find_it(json_data,ind,'nts',ind2,'glyco_bond')+delim

                  #f4 is puckering
                  f4=find_it(json_data,ind,'nts',ind2,'puckering')+delim
                  f4s=f4
                  f4l=f4


                  if out_more == 1:
                      fp.write(f1+f2+f3+f4l+eol)
                      if out_screen == 1:
                          print(f1+f2+f3+f4l)
                  else:
                      fp.write(f1+f2+f3+f4s+eol)
                      if out_screen == 1:
                          print(f1+f2+f3+f4s)
            fp.close()

            #code for json_detailedbp.py



            #output file name
            print("Processing ./detailedbp.txt")
            fname="./detailedbp.txt"
            #following outputs all data to file when =1,
            #smaller output when =0
            out_more=0

            #output to screen (=1) or silent (0)
            out_screen=0

            #field delimiter and eol
            delim="#"
            eol="\n"

            #open json file
            json_fname = "./merged.json"
            with open(json_fname) as json_file:
              json_data = json.load(json_file, strict=False)

            #names for column labels. should end with eol character
            #User must be sure this list is consistent with
            #value of out_more
            labels = ['sequence','nt1','nt2','hbond1','hbond2','hbond3', 'hbond4', 'hbond5']

            #open output file
            fp = open(fname, "w")
            #write dolumn labels
            for l in labels:
              fp.write(l+delim)
            fp.write(eol)


            results = []
            for ind in range(len(json_data)):
                  try:
                    f1=json_data[ind]['metadata']['str_id']+delim
                  except:
                    print("Error: ",json_data[ind]) # no nucleotides found
                    continue

                  for ind2 in dict_count(json_data[ind],'pairs'):
                      f2=find_it(json_data,ind,'pairs',ind2,'nt1')
                      f2 = re.sub("[^a-z0-9?.]+","", f2, flags=re.IGNORECASE) # 03/13/2023: ? for those chain id longer than 1 letter: must have this, 03/13/2023 find bugs when the line contains non-printable ascii characters \xXX  ..A.C.4.\x04
                      f2s=f2[len(f2)-2:len(f2)-1]+delim
                      f2l=f2+delim+f2s

                      f3=find_it(json_data,ind,'pairs',ind2,'nt2')
                      f3 = re.sub("[^a-z0-9?.]+","", f3, flags=re.IGNORECASE) # 03/13/2023: ? for those chain id longer than 1 letter: must have this, 03/13/2023 find bugs when the line contains non-printable ascii characters \xXX  ..A.C.4.\x04
                      f3s=f3[len(f3)-2:len(f3)-1]+delim
                      f3l=f3+delim+f3s

                      f4=find_it(json_data,ind,'pairs',ind2,'hbonds_desc')+delim
                      #remove [  ]
                      f4=re.sub(r'\[.*?\]','', f4)
                      #break at comma into two strings
                      f4=re.sub(',', delim, f4)

                      if out_more == 1:
                          fp.write(f1+f2l+f3l+f4+eol)
                          if out_screen == 1:
                              print(f1+f2l+f3l+f4)
                      else:
                          fp.write(f1+f2s+f3s+f4+eol)
                          if out_screen == 1:
                              print(f1+f2s+f3s+f4)

            fp.close()

            ## create

            import csv
            import pandas as pd

            ##csv_file = "stacks.csv"
            ##txt_1 = csv.reader(open('stacks.txt', "r"), delimiter = '#')
            txt_1 = pd.read_csv('stacks.txt', sep='#')
            #out_csv = csv.writer(open(csv_file, 'wb'))
            ##out_csv = csv.writer(open(csv_file, 'w'))
            ##out_csv.writerows(in_txt)
            txt_1.to_csv("stacks.csv")
            #print (txt_1)

            ##csv_file = "conf_pucker.csv"
            ##txt_2 = csv.reader(open('conf_pucker.txt', "r"), delimiter = '#')
            txt_2 = pd.read_csv('conf_pucker.txt', sep='#')
            #out_csv = csv.writer(open(csv_file, 'wb'))
            ##out_csv = csv.writer(open(csv_file, 'w'))
            ##out_csv.writerows(i_txt)
            txt_2.to_csv("conf_pucker.csv")
            #print (txt_2)

            ##csv_file = "hbonds.csv"
            ##txt_3 = csv.reader(open('hbonds.txt', "r"), delimiter = '#')
            txt_3 = pd.read_csv('hbonds.txt', sep='#')
            #out_csv = csv.writer(open(csv_file, 'wb'))
            ##out_csv = csv.writer(open(csv_file, 'w'))
            ##out_csv.writerows(txt)
            txt_3.to_csv("hbonds.csv")
            #print (txt_3)

            ##csv_file = "bp.csv"
            ##txt_4= csv.reader(open('bp.txt', "r"), delimiter = '#')
            txt_4 = pd.read_csv('bp.txt', sep='#')
            #out_csv = csv.writer(open(csv_file, 'wb'))
            ##out_csv = csv.writer(open(csv_file, 'w'))
            ##out_csv.writerows(txt)
            txt_4.to_csv("bp.csv")
            #print (txt_4)

            ##csv_file = "detailedbp.csv"
            ##txt_5 = csv.reader(open('detailedbp.txt', "r"), delimiter = '#')
            txt_5 = pd.read_csv('detailedbp.txt', sep= '#')
            #out_csv = csv.writer(open(csv_file, 'wb'))
            ##out_csv = csv.writer(open(csv_file, 'w'))
            ##out_csv.writerows(txt)
            txt_5.to_csv("detailedbp.csv")
            #print (txt_5)


            ## count
            cwd = os.path.split(os.getcwd())[1]
            cutoff = os.path.split(os.getcwd())[0]
            cutoff1 = os.path.basename(os.path.normpath(cutoff))
            data_list = re.findall(r'\d+(?:\.\d+)?', cutoff1)
            cutoff2 = str(data_list).strip('[]')
            cutoff3 = cutoff2.replace("'", "")

            now = datetime.datetime.now()

            print('Processing stacks.csv....')
            d = pd.read_csv('stacks.csv')
            unique_sequences = list(d['sequence'].unique())

            #total = len(unique_sequences)
            total = len(pdb_files) # 05/18/2023: We concluded that the denominator should be the total number of structure files.


            print ('\n')
            print ("Date and time of run:", now.strftime("%Y-%m-%d %H:%M"))
            print ('Degenerate Seqence:', cwd)
            print ('Total Number of Structures:', total)
            print  ('RMSD Cutoff:', cutoff3,'A')
            print ('\n')

            nt1a = list(d['nt1'])
            ntlength = len(nt1a)

            if ntlength == 0:
              print ('No stacking interactions found.')

            if ntlength != 0:
              d['nts_stacking'] = d[['nt1','nt2','stacking']].astype(str).apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)
              stacks = list(d['nts_stacking'])
              P = Counter(stacks)

              stacksab = list(d['nts_stacking'])
              stacksc = [x for x in stacksab if not 'not included' in x]

              P = Counter(stacksc)
              for x in P:
                  P[x]/=total

              for x in P:
                  P[x]*=100

              headers = ['NT 1, NT 2 & Stacking', 'Percentages']
              #data = sorted([(k,v) for k,v in P.iteritems() if v != 0])
              data = sorted([(k,v) for k,v in P.items() if v != 0])
              print(tabulate(data, headers=headers, tablefmt='grid'))

              #da = pd.DataFrame.from_dict(P, orient='index', dtype=Counter)

              if len(P)!=0:  # 03/13/2023, added to avoid error when empty
                da = pd.DataFrame.from_dict(P, orient='index', dtype= None)
              else:
                da = pd.DataFrame(columns=["NT 1, NT 2 & Stacking", "Percentage of Structures with Stacking Interaction"])

              da.to_csv('stacks_percent.csv', header=False) # 2023/10/28, add , header=False
              dab = pd.read_csv('stacks_percent.csv', names = ["NT 1, NT 2 & Stacking", "Percentage of Structures with Stacking Interaction"])
            else:
              da = pd.DataFrame(columns=["NT 1, NT 2 & Stacking", "Percentage of Structures with Stacking Interaction"])
              da.to_csv('stacks_percent.csv', header=False) # 2023/11/06, add if empty stack
              dab = pd.read_csv('stacks_percent.csv', names = ["NT 1, NT 2 & Stacking", "Percentage of Structures with Stacking Interaction"])
            print ("\n")




            print('Processing conf_pucker.csv....')
            df = pd.read_csv('conf_pucker.csv')

            nt = list(df['nt'])

            df['nt_conf'] = df[['nt', 'base_conf']].astype(str).apply(lambda x : '{}{}'.format(x[0],x[1]), axis=1)

            conf = list(df['nt_conf'])

            conf1 = [x for x in conf if not 'nan' in x]
            conf2 = [x for x in conf1 if not 'blank' in x]



            ntmax = max(nt)

            while ntmax > 0:

              nt_count = nt.count(ntmax)
              conf_count = Counter(conf2)

              ntmax = ntmax - 1


              for x in conf_count:
                  conf_count[x] = conf_count[x]/nt_count
                  conf_count[x] = conf_count[x]*100



            headers1 = ['NT & Base Conformation', 'Percentages']
            data = sorted([(k,v) for k,v in conf_count.items() if v != 0])
            print(tabulate(data, headers=headers1, tablefmt='grid'))
            print ('\n')



            dir_path = os.getcwd

            df['nt_pucker'] = df[['nt','puckering']].astype(str).apply(lambda x : '{}{}'.format(x[0],x[1]), axis=1)

            pucker = list(df['nt_pucker'])

            pucker1 = [x for x in pucker if not 'nan' in x]
            pucker2 = [x for x in pucker1 if not 'blank' in x]

            ntmax1 = max(nt)

            while ntmax1 > 0:
              nt_count1 = nt.count(ntmax1)
              pucker_count = Counter(pucker2)

              ntmax1 = ntmax1 - 1

              for x in pucker_count:
                  pucker_count[x] = pucker_count[x]/nt_count1
                  pucker_count[x] = pucker_count[x]*100

            headers2 = ['NT & Sugar Pucker', 'Percentages']
            data = sorted([(k,v) for k,v in pucker_count.items() if v != 0])
            print(tabulate(data, headers=headers2, tablefmt='grid'))
            print ('\n')


            #dfb = pd.DataFrame.from_dict(conf_count, orient='index', dtype=Counter)
            if len(conf_count)!=0:  # 03/13/2023, added to avoid error when empty
              dfb = pd.DataFrame.from_dict(conf_count, orient='index', dtype=None)
            else:
              dfb = pd.DataFrame(columns=["NT & Base Conformation", "Percentage of NTs with Conformation"])


            dfb.to_csv('conf_percent.csv', header=False) # 2023/10/28, add , header=False
            dfbb = pd.read_csv('conf_percent.csv', names = ["NT & Base Conformation", "Percentage of NTs with Conformation"])

            #dfa = pd.DataFrame.from_dict(pucker_count, orient='index', dtype=Counter)
            if len(pucker_count)!=0:  # 03/13/2023, added to avoid error when empty
              dfa = pd.DataFrame.from_dict(pucker_count, orient='index', dtype=None)
            else:
              dfa = pd.DataFrame(columns=["NT & Sugar Pucker", "Percentage of NTs with Pucker"])

            dfa.to_csv('pucker_percent.csv', header=False) # 2023/10/28, add , header=False
            dfab = pd.read_csv('pucker_percent.csv', names = ["NT & Sugar Pucker", "Percentage of NTs with Pucker"])


            print('Processing hbonds.csv....')
            #df1 = pd.read_csv('hbonds.csv',error_bad_lines=False)
            df1 = pd.read_csv('hbonds.csv')

            sequences = list(df1['sequence'].unique())

            #total = len(sequences)
            total = len(pdb_files) # 05/18/2023: We concluded that the denominator should be the total number of structure files.

            print ('\n')
            print ("Date and time of run:", now.strftime("%Y-%m-%d %H:%M"))
            print ('Degenerate Seqence:', cwd)
            print ('Total Number of Structures:', total)
            print  ('RMSD Cutoff:', cutoff3,'A')
            print ('\n')


            nts_hbond1a = list(df1['hbond1'])
            totalhbonds = len(nts_hbond1a)

            if totalhbonds == 0:
              print ('No non-pairing hydrogen bonding interactions found.')

            if totalhbonds != 0:
              hbond1 = {k: v for v, k in enumerate(nts_hbond1a)}
              df1['nts_hbond1'] = df1[['nt1','nt2','hbond1']].astype(str).apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)
              df1['nts_hbond2'] = df1[['nt1','nt2','hbond2']].astype(str).apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)
              df1['nts_hbond3'] = df1[['nt1','nt2','hbond3']].astype(str).apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)
              df1['nts_hbond4'] = df1[['nt1','nt2','hbond4']].astype(str).apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)

              hbonds0 = list(df1['nts_hbond1'])
              hbonds1 = list(df1['nts_hbond2'])
              hbonds2 = list(df1['nts_hbond3'])
              hbonds3 = list(df1['nts_hbond4'])

              hbond = itertools.chain(hbonds0, hbonds1, hbonds2, hbonds3)
              hbond_list = list(hbond)


              hbonds = [x for x in hbond_list if not 'nan' in x]
              hbonds1a = [x for x in hbonds if not 'blank' in x]
              count16 = Counter(hbonds1a)

              for x in count16:
                  count16[x]/=total

              for x in count16:
                  count16[x]*=100

              headers3 = ['NT 1, NT 2, and Hydrogen Bond', 'Percentages']
              data = sorted([(k,v) for k,v in count16.items() if v !=0])
              print(tabulate(data, headers=headers3, tablefmt='grid'))

              #df1a = pd.DataFrame.from_dict(count16, orient='index', dtype=Counter)
              if len(count16)!=0: # 03/13/2023, added to avoid error when empty
                df1a = pd.DataFrame.from_dict(count16, orient='index', dtype=None)
              else:
                df1a= pd.DataFrame(columns=["NT 1, NT 2 & Hydrogen Bond", "Percentage of Structures with Non-Pairing HB"])
              df1a.to_csv('hbond_percent.csv', header=False) # 2023/10/28, add , header=False
              df1ab = pd.read_csv('hbond_percent.csv', names = ["NT 1, NT 2 & Hydrogen Bond", "Percentage of Structures with Non-Pairing HB"])
            else:
              df1a= pd.DataFrame(columns=["NT 1, NT 2 & Hydrogen Bond", "Percentage of Structures with Non-Pairing HB"])
              df1a.to_csv('hbond_percent.csv', header=False) # 2023/11/06, add if empty hydrogen
              df1ab = pd.read_csv('hbond_percent.csv', names = ["NT 1, NT 2 & Hydrogen Bond", "Percentage of Structures with Non-Pairing HB"])

            print ('\n')

            print('Processing bp.csv....')
            df3 = pd.read_csv('bp.csv')
            sequences_bp = list(df3['sequence'].unique())

            #totals = len(sequences)
            totals = len(pdb_files) # 05/18/2023: We concluded that the denominator should be the total number of structure files.

            print ('\n')
            print ("Date and time of run:", now.strftime("%Y-%m-%d %H:%M"))
            print ('Degenerate Seqence:', cwd)
            print ('Total Number of Structures:', totals)
            print  ('RMSD Cutoff:', cutoff3,'A')
            print ('\n')

            nt1aa = list(df3['nt1'])

            totalbp = len(nt1aa)

            if totalbp == 0:
              print ('No base pairing interactions found.')
            if totalbp != 0:
              df3['nts_bp'] = df3[['nt1','nt2']].apply(lambda x : '{}{}'.format(x[0],x[1]), axis=1)

              pairs = list(df3['nts_bp'])

              pairs1 = [x for x in pairs if not 'nan' in x]
              pairs2 = [x for x in pairs1 if not 'blank' in x]

              counta = Counter(pairs2)


              for x in counta:
                  counta[x]/=totals
                  counta[x]*=100


              headers4 = ['Paired Nucleotides', 'Percentages']
              #data = sorted([(k,v) for k,v in counta.iteritems() if v != 0])
              data = sorted([(k,v) for k,v in counta.items() if v != 0])
              print(tabulate(data, headers=headers4, tablefmt='grid'))
              print ('\n')


              df3['nts_bp_type'] = df3[['nt1','nt2','name']].apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)

              pair = list(df3['nts_bp_type'])

              pair1 = [x for x in pair if not 'nan' in x]
              pair2 = [x for x in pair1 if not 'blank' in x]

              count1 = Counter(pair2)


              for x in count1:
                  count1[x]/=totals
                  count1[x]*=100


              headers5 = ['NT 1, NT 2 & Base Pair Classification', 'Percentages']
              #data = sorted([(k,v) for k,v in count1.iteritems() if v != 0])
              data = sorted([(k,v) for k,v in count1.items() if v != 0])
              print(tabulate(data, headers=headers5, tablefmt='grid'))

              #df3b = pd.DataFrame.from_dict(counta, orient='index', dtype=Counter)
              if len(counta)!=0:  # 03/13/2023, added to avoid error when empty
                df3b = pd.DataFrame.from_dict(counta, orient='index', dtype=None)
              else:
                df3b = pd.DataFrame(columns=["Paired Nucleotides", "Percentage of Structures with Pairing"])

              #df3b = pd.DataFrame.from_dict(counta, orient='index', dtype=None)
              df3b.to_csv('bp_percent.csv', header=False) # 2023/10/28, add , header=False
              df3bb = pd.read_csv('bp_percent.csv', names = ["Paired Nucleotides", "Percentage of Structures with Pairing"])


              #df3a = pd.DataFrame.from_dict(count1, orient='index', dtype=Counter)
              if len(count1)!=0:  # 03/13/2023, added to avoid error when empty
                df3a = pd.DataFrame.from_dict(count1, orient='index', dtype=None)
              else:
                df3a = pd.DataFrame(columns=["NT 1, NT 2 & Base Pair Classification", "Percentage of Base Pair Classification"])

              #df3a = pd.DataFrame.from_dict(count1, orient='index', dtype=None)
              df3a.to_csv('bpclass_percent.csv', header=False) # 2023/10/28, add , header=False
              df3ab = pd.read_csv('bpclass_percent.csv', names = ["NT 1, NT 2 & Base Pair Classification", "Percentage of Base Pair Classification"])
            else:
              df3a = pd.DataFrame(columns=["NT 1, NT 2 & Base Pair Classification", "Percentage of Base Pair Classification"])
              df3a.to_csv('bpclass_percent.csv', header=False) # 2023/11/06, add if empty bp-pair
              df3ab = pd.read_csv('bpclass_percent.csv', names = ["NT 1, NT 2 & Base Pair Classification", "Percentage of Base Pair Classification"])

            print ('\n')




            print('Processing detailedbp.csv....')
            #df4 = pd.read_csv('detailedbp.csv',error_bad_lines=False)
            df4 = pd.read_csv('detailedbp.csv')

            sequences = list(df1['sequence'].unique())


            #totalb = len(sequences)
            totalb = len(pdb_files) # 05/18/2023: We concluded that the denominator should be the total number of structure files.

            print ('\n')
            print ("Date and time of run:", now.strftime("%Y-%m-%d %H:%M"))
            print ('Degenerate Seqence:', cwd)
            print ('Total Number of Structures:', totalb)
            print  ('RMSD Cutoff:', cutoff3,'A')
            print ('\n')


            hbond1aa = list(df4['nt1'])
            totalbphb = len(hbond1aa)

            if totalbphb != 0:
              hbond1 = {k: v for v, k in enumerate(hbond1aa)}
              df4['nts_hbond1'] = df4[['nt1','nt2','hbond1']].apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)
              df4['nts_hbond2'] = df4[['nt1','nt2','hbond2']].apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)
              df4['nts_hbond3'] = df4[['nt1','nt2','hbond3']].apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)
              df4['nts_hbond4'] = df4[['nt1','nt2','hbond4']].apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)
              df4['nts_hbond5'] = df4[['nt1','nt2','hbond5']].apply(lambda x : '{}{}{}'.format(x[0],x[1],x[2]), axis=1)


              keywordFilter = set(['blank'])

              hbonds3 = list(df4['nts_hbond1'])
              hbonds4 = list(df4['nts_hbond2'])
              hbonds5 = list(df4['nts_hbond3'])
              hbonds6 = list(df4['nts_hbond4'])
              hbonds7 = list(df4['nts_hbond5'])



              hbond0 = itertools.chain(hbonds3, hbonds4, hbonds5, hbonds6, hbonds7)
              hbond_lists = list(hbond0)



              hbonds1 = [x for x in hbond_lists if not 'nan' in x]
              hbonds2 = [x for x in hbonds1 if not 'blank' in x]

              countb = Counter(hbonds2)

              for x in countb:
                  countb[x]/=totalb

              for x in countb:
                  countb[x]*=100

              headers6 = ['NT 1, NT 2 & Pairing Hydrogen Bonds', 'Percentages']
              #data = sorted([(k,v) for k,v in countb.iteritems() if v != 0])
              data = sorted([(k,v) for k,v in countb.items() if v != 0])
              print(tabulate(data, headers=headers6, tablefmt='grid'))

              #df4a = pd.DataFrame.from_dict(countb, orient='index', dtype=Counter)
              if len(countb)!=0:  # 03/13/2023, added to avoid error when empty
                df4a = pd.DataFrame.from_dict(countb, orient='index', dtype=None)
              else:
                df4a = pd.DataFrame(columns=["NT 1, NT 2 & Pairing Hydrogen Bonds", "Percentage of Pairing Hydrogen Bonds"])


              #df4a = pd.DataFrame.from_dict(countb, orient='index', dtype=None)
              df4a.to_csv('phbond_percent.csv', header=False) # 2023/10/28, add , header=False
              df4ab = pd.read_csv('phbond_percent.csv', names = ["NT 1, NT 2 & Pairing Hydrogen Bonds", "Percentage of Pairing Hydrogen Bonds"])
            else:
              df4a = pd.DataFrame(columns=["NT 1, NT 2 & Pairing Hydrogen Bonds", "Percentage of Pairing Hydrogen Bonds"])
              df4a.to_csv('phbond_percent.csv', header=False) # 2023/11/06, add if empty bp-pair
              df4ab = pd.read_csv('phbond_percent.csv', names = ["NT 1, NT 2 & Pairing Hydrogen Bonds", "Percentage of Pairing Hydrogen Bonds"])

            print ('\n')

            if totalbp != 0:
              df3bb['Paired Nucleotides'] = df3bb['Paired Nucleotides'].astype(str)
              total_frames = pd.concat([dab, dfbb, dfab, df1ab, df3bb, df3ab, df4ab], axis=1)
              total_frames.to_csv('summary1.csv', index=None)

              first_row_num = 1
              #delete_row = {2}
              delete_row = {} # 2023/11/1 update. The previous table has been processed to exclude na row.
              with open('summary1.csv', 'rt') as infile, open('summary.csv', 'wt') as outfile:
                  outfile.writelines(row for row_num, row in enumerate(infile, first_row_num)
                                if row_num not in delete_row)
            print("done!")

            os.chdir(cur_work_dir)
    except:
        print("Receving errors, report errors to GitHub!")
        print("Back to ",cur_work_dir)
        os.chdir(cur_work_dir)
    print("Back to ",cur_work_dir)
    os.chdir(cur_work_dir)




In [ ]:
#run DSSR on folder containing the good clips
PDB_folder = 'good/'



blacklist = []
original	 = True 
clips_og	 = True 
clips	 = True 
bad_clips	 = True 

blacklist = []
if original:
  blacklist.append('original')
if clips_og:
  blacklist.append('clips_og')
if clips:
  blacklist.append('clips')
if bad_clips:
  blacklist.append('bad_clips')



#process_dir = cur_work_dir +'/good_clips/best-fits/'
process_dir = work_dir

print("!!!! Setting up blacklist: ",blacklist)
dssr_characterization(process_dir, blacklist)

!!!! Setting up blacklist:  ['original', 'clips_og', 'clips', 'bad_clips']

******************************************************************
              DSSR: an Integrated Software Tool for
             Dissecting the Spatial Structure of RNA
              v2.3.2-2021jun29 by xiangjun@x3dna.org

As of version 2, DSSR may be LICENSED from Columbia University.
DSSR basic is FREE for academic uses, with ABSOLUTELY NO WARRANTY.
DSSR Pro is available for paid academic or commercial users only,
and it is actively maintained and continuously improved.

This is DSSR *basic*. Advanced features are available in DSSR Pro.
******************************************************************

Time used: 00:00:00:00
Entering  /content/drive/MyDrive/AML/AML_Project/Hairpin_4
!!!! Skipping folder  /content/drive/MyDrive/AML/AML_Project/Hairpin_4/clips_og
!!!! Skipping folder  /content/drive/MyDrive/AML/AML_Project/Hairpin_4/clips
!!!！ Processing family:  /content/drive/MyDrive/AML/AML_Project/Hairp